<h2>File paths and imports</h2>

This steps is important because it tells the program where it can access the files needed throughout the process.

You must specify 3 paths:

<li> <b>model_path</b>: body detection model
<li> <b>input_video_directory</b>: Folder containing the input videos
<li> <b>output_video_directory</b>: Folder where the treated videos will be redirected to

You can also mention videos that you don't want to be treated. To do so, simply indicate their name(s) without the extension in <b>ignore_S1</b> and <b>ignore_S2</b>.

In [ ]:
from ui_lib_strongSort import *

# path of the body detection model
body_model_path = "/etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/Code/Tracking/strong_sort/Body_detection_models/Body_detection_model.pt"

# video paths:
# input video directory (without any annotation)
input_video_directory = "/etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/ChimpRec_videos/Input"
# output (final version - with human interaction)
output_video_directory = "/etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/ChimpRec_videos/Output"

ignore_S1 = [ # related to step 1
    "1",
    "2",
    "3",
    "4",
    #"4_cropped",
    "5",
    "6",
    "7",
    "8",
    "12h41_full.MP4",
    "12h41_short.MP4"
]

ignore_S2 = [ # related to step 2
    "1",
    "2",
    "3",
    "4",
    #"4_cropped",
    "5",
    "6",
    "7",
    "8",
    "12h41_full.MP4",
    "12h41_short.MP4"
]



2026-02-05 12:07:31.992 | MainProcess/MainThread | INFO     | /etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/.venv/lib64/python3.10/site-packages/boxmot/trackers/basetracker.py:56 | __init__ - BaseTracker initialization parameters:
2026-02-05 12:07:31.992 | MainProcess/MainThread | INFO     | /etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/.venv/lib64/python3.10/site-packages/boxmot/trackers/basetracker.py:57 | __init__ - det_thresh: 0.3
2026-02-05 12:07:31.992 | MainProcess/MainThread | INFO     | /etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/.venv/lib64/python3.10/site-packages/boxmot/trackers/basetracker.py:58 | __init__ - max_age: 150
2026-02-05 12:07:31.992 | MainProcess/MainThread | INFO     | /etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/.venv/lib64/python3.10/site-packages/boxmot/trackers/basetracker.py:59 | __init__ - max_obs: 50
2026-02-05 12:07:31.992 | MainProcess/MainThread | INFO     | /etinfo/users/2024/trixen/Documents/Ma

<h2>First step:</h2>

This step will process the input videos automatically. In other words, it will draw rectangles around each individuals and track them throughout the video.

In [2]:
# FIRST STEP CODE (double-click to extend)

# Directories (same as before)
mannual_annotations_directory = f"{input_video_directory}/manual_annotations"
output_video_directory_temp = f"{output_video_directory}/temp"
raw_text_output_directory = f"{output_video_directory_temp}/raw_output"

# Create the directories if they do not exist yet
os.makedirs(input_video_directory, exist_ok=True)
os.makedirs(output_video_directory, exist_ok=True)
os.makedirs(mannual_annotations_directory, exist_ok=True)
os.makedirs(output_video_directory_temp, exist_ok=True)
os.makedirs(raw_text_output_directory, exist_ok=True)

# YOLOv8s initialisation (same)
YOLOv8s = YOLO(body_model_path)

# StrongSORT initialisation (no manual OSNet or DeepSORT metric needed)
# If you have a reid weights .pt, point to it; otherwise set to None and StrongSORT may auto-handle/download defaults
reid_weights_path = "/etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/Code/Tracking/strong_sort/Re-ID_models/osnet_ain_x1_0_imagenet.pth"  # e.g. "/path/to/osnet_x0_25_msmt17.pt" if you have one
configuration_file_path = "/etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/Code/Tracking/strong_sort/config/strongsort_config.yaml"  # e.g. "/path/to/strong_sort.yaml" if you have a custom config; otherwise set to None
device = 'cuda' if torch.cuda.is_available() else 'cpu'
strongsort = build_strongsort(reid_weights=reid_weights_path, device=device, fp16=False, tracker_config_path=configuration_file_path)

for input_video in os.listdir(input_video_directory):
    if input_video.endswith(".mp4") or input_video.endswith(".MP4"):

        full_video_path = os.path.join(input_video_directory, input_video)
        video_name = os.path.splitext(input_video)[0]

        if video_name in ignore_S1:
            print(f"{video_name}.mp4 ignored")
            continue

        # creation of the manual_annotation textfile if it doesn't exist yet
        annotation_file_path = f"{mannual_annotations_directory}/{video_name}.txt"
        try:
            with open(annotation_file_path, 'x') as f:
                print(f"{video_name}.txt automatically created in {mannual_annotations_directory}.")
        except FileExistsError:
            print(f"{video_name}.txt already present in {mannual_annotations_directory}.")
        print("\n")

        # production of the textual outputs with StrongSORT
        perform_tracking(
            input_video_path = full_video_path, 
            output_text_file_path = f"{raw_text_output_directory}/{video_name}.txt", 
            detection_model = YOLOv8s, 
            tracker = strongsort,
            confidence_threshold = 0.5
        )
        print(f"Annotations ready for video: {full_video_path}.\n")

        # production of the visual output (unchanged)
        draw_bbox_from_file(
            file_path = f"{raw_text_output_directory}/{video_name}.txt", 
            input_video_path = full_video_path, 
            output_video_path = f"{output_video_directory_temp}/{video_name}-(temp).mp4",
            annotation_type="bbox",
            draw_frame_count=True
        )
        print(f"Treatment done: {full_video_path}.\n")

self.max_obs 155
1.mp4 ignored
3.mp4 ignored
2.mp4 ignored
4_cropped.txt automatically created in /etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/ChimpRec_videos/Input/manual_annotations.




Tracking progress (4_cropped.MP4):  98%|█████████▊| 1501/1537 [03:13<00:04,  7.74it/s]


Annotations ready for video: /etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/ChimpRec_videos/Input/4_cropped.MP4.



Drawing annotations (4_cropped.MP4):  98%|█████████▊| 1501/1537 [00:25<00:00, 58.82it/s]


Treatment done: /etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/ChimpRec_videos/Input/4_cropped.MP4.

4.mp4 ignored
5.mp4 ignored
6.mp4 ignored
7.mp4 ignored
8.mp4 ignored
12h41_short.txt already present in /etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/ChimpRec_videos/Input/manual_annotations.




...

Treatment done: /etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/ChimpRec_videos/Input/12h41_short.MP4.

12h41_full.txt already present in /etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/ChimpRec_videos/Input/manual_annotations.




Tracking progress (12h41_full.MP4): 100%|██████████| 10536/10536 [28:18<00:00,  6.20it/s]


Annotations ready for video: /etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/ChimpRec_videos/Input/12h41_full.MP4.



Drawing annotations (12h41_full.MP4): 100%|██████████| 10536/10536 [02:49<00:00, 62.17it/s]

Treatment done: /etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/ChimpRec_videos/Input/12h41_full.MP4.



<h2>Second step:</h2>

This final step will take into account your modifications to modify the output of the automated process.

<b>If you need to modify annotations previously created:</b> simply run this part of the code. In this case, there's no need to run the above cells.

In [6]:
#SECOND STEP CODE (double-click to extend)

for input_video in os.listdir(input_video_directory):
    if input_video.endswith(".mp4") or input_video.endswith(".MP4"):
        full_video_path = os.path.join(input_video_directory, input_video)
        video_name = os.path.splitext(input_video)[0]

        if video_name in ignore_S2:
            print(f"{video_name}.mp4 ignored")
            continue

        annotation_file = f"{mannual_annotations_directory}/{video_name}.txt"

        raw_reader = raw_tracking_data_reader(f"{raw_text_output_directory}/{video_name}.txt")

        try:
            edit_reader = modification_reader(annotation_file)
        except:
            print(f"Error: the manual annotation file related to the video <{full_video_path}> is not found. It must be located at <{annotation_file}>.")
            continue
        
        metadata_file_path = f"{output_video_directory}/{video_name}-treated.txt"
        output_video_path = f"{output_video_directory}/{video_name}-treated.mp4"
        writer = data_writer(metadata_file_path)

        # computation of the new metadata file
        modified_data = edit_raw_output(raw_reader, edit_reader) 

        # production of the textual output
        writer.write(modified_data)

        # production of the visual output
        draw_bbox_from_file(
            file_path = metadata_file_path, 
            input_video_path = full_video_path, 
            output_video_path = output_video_path,
            annotation_type="triangle"
        )
        print(f"Treatment done: {full_video_path}.\n")

1.mp4 ignored
3.mp4 ignored
2.mp4 ignored
4_cropped.mp4 ignored
4.mp4 ignored
5.mp4 ignored
6.mp4 ignored
7.mp4 ignored
8.mp4 ignored


Drawing annotations (12h41.MP4): 100%|█████████▉| 1732/1734 [00:20<00:00, 86.06it/s]

Treatment done: /etinfo/users/2024/trixen/Documents/Master_thesis/ChimpRec/ChimpRec_videos/Input/12h41.MP4.

